# Módulo 4 — Introdução à IA no Desenvolvimento

A inteligência artificial generativa está transformando a forma como analistas de dados trabalham. Este notebook explora como usar Claude Code no dia a dia de um profissional do setor elétrico.

## O que vamos ver

1. O que é Claude Code e para que serve
2. Como LLMs auxiliam na análise de dados
3. Fluxo de trabalho: prompt → revisão → execução
4. Demonstração: gerar função de qualidade com Claude
5. Dicas práticas para analistas do setor elétrico

---

> **Importante:** Claude Code é uma ferramenta de **assistência** — você revisa e valida todo o código gerado. Nunca execute código gerado por IA sem entendê-lo.

## 1. O que é Claude Code?

Claude Code é uma ferramenta CLI (Command Line Interface) da Anthropic que integra o modelo de linguagem Claude diretamente ao seu terminal e ambiente de desenvolvimento.

**Diferenças em relação ao ChatGPT/Claude.ai:**

| Característica | Claude.ai (web) | Claude Code (CLI) |
|----------------|-----------------|--------------------|
| Acesso a arquivos | Não (upload manual) | Sim, lê/escreve automaticamente |
| Contexto do projeto | Não | Sim, via CLAUDE.md |
| Executa comandos | Não | Sim (com aprovação) |
| Integração IDE | Não | Sim (VS Code, etc.) |
| Uso típico | Chat, perguntas | Desenvolvimento ativo |

---

**Para instalar:**
```bash
npm install -g @anthropic-ai/claude-code
export ANTHROPIC_API_KEY="sua-chave-aqui"
claude  # inicia a sessão interativa
```

## 2. Como LLMs Auxiliam na Análise de Dados

Grandes Modelos de Linguagem (LLMs) são excelentes para tarefas que envolvem texto e padrões — e código é uma forma de texto com padrões muito regulares.

In [ ]:
# Onde LLMs ajudam vs. onde você ainda precisa de julgamento humano
tarefas = {
    "LLMs são excelentes": [
        "Gerar código boilerplate (conexão, leitura de arquivo, estrutura de função)",
        "Escrever docstrings e documentação técnica",
        "Converter SQL em código Python pandas",
        "Explicar erros e sugerir correções",
        "Refatorar código para melhorar legibilidade",
        "Gerar queries Snowflake a partir de descrição em português",
        "Criar testes unitários para funções existentes",
    ],
    "Ainda precisam de julgamento humano": [
        "Definir as regras de negócio corretas (o que é um 'consumo anormal'?)",
        "Validar se os resultados fazem sentido para o setor",
        "Decisões sobre dados sensíveis (CPF, localização)",
        "Aprovar mudanças em sistemas de produção",
        "Interpretar anomalias que requerem conhecimento de campo",
    ]
}

for categoria, itens in tarefas.items():
    print(f"\n{categoria}:")
    for item in itens:
        print(f"  • {item}")

## 3. Fluxo de Trabalho com Claude Code

O fluxo recomendado é:

```
1. DESCREVER → Explique o problema em linguagem natural
2. REVISAR   → Leia o código gerado com atenção
3. TESTAR    → Execute em dados pequenos primeiro
4. VALIDAR   → Confira os resultados com o negócio
5. ITERAR   → Peça ajustes se necessário
```

Nunca pule a etapa de revisão, especialmente para queries que modificam dados.

## 4. Demonstração: Função Gerada com Auxílio de Claude

O código abaixo foi escrito com o apoio do Claude Code a partir do seguinte prompt:

```
Crie uma função Python que recebe um DataFrame pandas com as colunas
cod_consumidor, nom_consumidor, cpf_cnpj, cod_modalidade_tarifaria,
cod_classe_consumo e vlr_consumo_medio_kwh, e retorna um dict com:
- resumo: contagem total, ativos, inativos
- completude: % de preenchimento por campo obrigatório
- problemas_documento: contagem de CPF/CNPJ ausentes ou inválidos
- outliers_consumo: lista de cod_consumidor com consumo > 3 desvios padrão

Use type hints, docstring em português e trate possíveis erros.
```

In [ ]:
# Código gerado com apoio do Claude Code (revisado e validado pelo analista)
import pandas as pd
import numpy as np
import re
from typing import Dict, Any, List


def analisar_qualidade_cadastro(df: pd.DataFrame) -> Dict[str, Any]:
    """
    Realiza análise de qualidade do cadastro de consumidores.
    
    Args:
        df: DataFrame com colunas do cadastro de UCs.
            Colunas esperadas: cod_consumidor, nom_consumidor, cpf_cnpj,
            cod_modalidade_tarifaria, cod_classe_consumo, vlr_consumo_medio_kwh
    
    Returns:
        Dicionário com análises de:
        - resumo: contagens gerais
        - completude: % de preenchimento por campo
        - problemas_documento: situação do CPF/CNPJ
        - outliers_consumo: registros com consumo anormal
    
    Raises:
        ValueError: se o DataFrame estiver vazio ou faltar colunas obrigatórias
    """
    # Validar entrada
    colunas_obrigatorias = [
        "cod_consumidor", "nom_consumidor", "cpf_cnpj",
        "cod_modalidade_tarifaria", "cod_classe_consumo", "vlr_consumo_medio_kwh"
    ]
    colunas_faltantes = [c for c in colunas_obrigatorias if c not in df.columns]
    if colunas_faltantes:
        raise ValueError(f"Colunas ausentes no DataFrame: {colunas_faltantes}")
    
    if df.empty:
        raise ValueError("DataFrame está vazio")
    
    total = len(df)
    
    # 1. Resumo geral
    ativos = df["flg_ativo"].sum() if "flg_ativo" in df.columns else None
    resumo = {
        "total_registros": total,
        "ativos": int(ativos) if ativos is not None else "coluna flg_ativo ausente",
        "inativos": int(total - ativos) if ativos is not None else "coluna flg_ativo ausente",
    }
    
    # 2. Completude por campo
    completude = {
        col: round(
            (df[col].notna() & (df[col].astype(str).str.strip() != "")).sum() / total * 100, 1
        )
        for col in colunas_obrigatorias
    }
    
    # 3. Problemas no documento (CPF/CNPJ)
    def classificar_doc(doc):
        if pd.isna(doc) or str(doc).strip() == "":
            return "ausente"
        digitos = re.sub(r'\D', '', str(doc))
        if len(digitos) in (11, 14):
            if digitos == digitos[0] * len(digitos):
                return "invalido_sequencia"
            return "valido"
        return "formato_invalido"
    
    tipos_doc = df["cpf_cnpj"].apply(classificar_doc).value_counts().to_dict()
    problemas_documento = {
        "validos": tipos_doc.get("valido", 0),
        "ausentes": tipos_doc.get("ausente", 0),
        "formato_invalido": tipos_doc.get("formato_invalido", 0),
        "invalido_sequencia": tipos_doc.get("invalido_sequencia", 0),
        "pct_problemas": round(
            (1 - tipos_doc.get("valido", 0) / total) * 100, 1
        )
    }
    
    # 4. Outliers de consumo (método: z-score > 3)
    consumo = df["vlr_consumo_medio_kwh"].dropna()
    if len(consumo) > 3:
        media = consumo.mean()
        std = consumo.std()
        z_scores = (df["vlr_consumo_medio_kwh"] - media) / std
        mask_outlier = z_scores.abs() > 3
        outliers = df.loc[mask_outlier, ["cod_consumidor", "nom_consumidor", "vlr_consumo_medio_kwh"]]
        outliers_consumo = outliers.to_dict(orient="records")
    else:
        outliers_consumo = []
    
    return {
        "resumo": resumo,
        "completude": completude,
        "problemas_documento": problemas_documento,
        "outliers_consumo": outliers_consumo
    }


# Testar com os dados do curso
# Para garantir que o arquivo seja encontrado independente de onde o Jupyter foi iniciado
import os
possible_paths = [
    "../modulo2_dados_com_pandas/dados/consumidores_simulado.csv", # Se o Jupyter rodar dentro da pasta modulo4
    "modulo2_dados_com_pandas/dados/consumidores_simulado.csv"    # Se o Jupyter rodar na raiz do projeto
]

file_path = next((p for p in possible_paths if os.path.exists(p)), None)

if not file_path:
    raise FileNotFoundError("Arquivo de dados não encontrado. Verifique a estrutura de pastas.")

df = pd.read_csv(file_path)


resultado = analisar_qualidade_cadastro(df)

print("=== Análise de Qualidade do Cadastro ===")
print("\n--- Resumo ---")
for k, v in resultado["resumo"].items():
    print(f"  {k}: {v}")

print("\n--- Completude (%) ---")
for campo, pct in resultado["completude"].items():
    barra = "█" * int(pct / 5)
    print(f"  {campo:<35}: {pct:>5.1f}% {barra}")

print("\n--- Documentos ---")
for k, v in resultado["problemas_documento"].items():
    print(f"  {k}: {v}")

print(f"\n--- Outliers de Consumo ({len(resultado['outliers_consumo'])}) ---")
for out in resultado["outliers_consumo"]:
    print(f"  UC {out['cod_consumidor']}: {out['nom_consumidor']} — {out['vlr_consumo_medio_kwh']:,.1f} kWh")


## 4.5 Demonstração Avançada: Agentes e Function Calling com Gemini

A grande evolução da IA para o desenvolvimento é a capacidade dos modelos interagirem diretamente com o seu código (Agentic Workflow). O Google Gemini 2.5, por exemplo, permite o uso de **Ferramentas (Tools/Function Calling)**.

Nesse cenário, nós escrevemos uma função em Python com regras de negócio rígidas (ex: classificação tarifária). A IA não precisa 'adivinhar' a regra; ela aciona a sua função Python localmente, recebe o resultado determinístico e então formula a resposta para o usuário final.

> **Pré-requisito:** Instale a biblioteca do Gemini (`uv pip install google-genai`) e configure a variável de ambiente `GEMINI_API_KEY`.

In [ ]:
import os
from dotenv import load_dotenv

# Carrega as variáveis de ambiente do arquivo .env
load_dotenv()

from google import genai
from google.genai import types

# 1. Definimos uma função determinística com a regra de negócio do setor elétrico
def classificar_anomalia_consumo(kwh_atual: float, kwh_media_historica: float) -> str:
    """
    Classifica se o consumo atual de uma unidade apresenta anomalia.
    
    Args:
        kwh_atual: O consumo medido no mês atual (em kWh).
        kwh_media_historica: A média de consumo dos últimos 12 meses (em kWh).
        
    Returns:
        Uma string indicando a classificação: 'NORMAL', 'ANOMALIA_ALTA', ou 'ANOMALIA_BAIXA'.
    """
    if kwh_media_historica == 0:
        return "NORMAL"
        
    variacao = (kwh_atual - kwh_media_historica) / kwh_media_historica
    
    if variacao > 0.40:
        return "ANOMALIA_ALTA"  # Possível gato ou equipamento defeituoso
    elif variacao < -0.40:
        return "ANOMALIA_BAIXA" # Possível medidor travado ou imóvel desocupado
    
    return "NORMAL"

# 2. Iniciamos o cliente do Gemini (garanta que tem a variável GEMINI_API_KEY configurada)
# Para o notebook rodar limpo caso não tenha a chave ainda, fazemos um try/except
try:
    # Instanciar o cliente
    client = genai.Client()
    
    print("Analisando o cenário com o Gemini (usando a ferramenta local)...")
    
    # 3. Pedimos para a IA analisar um caso real. Note que passamos a nossa função Python na lista de tools.
    response = client.models.generate_content(
        model='gemini-3.1-flash-lite-preview',
        contents="O cliente João teve um consumo de 520 kWh neste mês. Historicamente, a média dele era de 210 kWh. Classifique o status deste consumo e redija um e-mail formal para a equipe de fiscalização de campo caso haja anomalia.",
        config=types.GenerateContentConfig(
            tools=[classificar_anomalia_consumo], # <-- O poder do Function Calling!
            temperature=0.2
        )
    )
    
    print("\n--- RESPOSTA DA IA ---")
    print(response.text)
    
except Exception as e:
    print(f"Não foi possível executar a chamada ao Gemini. Verifique sua GEMINI_API_KEY.\nErro: {e}")

### 4.6 Exemplo de Agente Analisando a Base de Dados Inteira

Nesta etapa, vamos criar ferramentas (tools) que permitem ao Gemini 'enxergar' o nosso DataFrame. Em vez de passarmos apenas uma string com dados de um cliente, damos à IA o poder de perguntar ao Python sobre qualquer registro na base.

In [ ]:
# 1. Definimos funções que servirão de interface entre o Gemini e o nosso DataFrame

def buscar_dados_consumidor(cod_consumidor: int) -> str:
    """
    Busca as informações de um consumidor na base de dados pelo seu código.
    """
    # O df deve estar carregado globalmente no notebook para esta função funcionar
    registro = df[df['cod_consumidor'] == cod_consumidor]
    if registro.empty:
        return f"Consumidor {cod_consumidor} não encontrado."
    return registro.to_json(orient='records')

def listar_outliers_projeto() -> list:
    """
    Retorna uma lista de códigos de consumidores que possuem consumo acima da média da base.
    """
    media = df['vlr_consumo_medio_kwh'].mean()
    outliers = df[df['vlr_consumo_medio_kwh'] > media * 2]['cod_consumidor'].tolist()
    return outliers

# 2. Chamada do Agente com Múltiplas Ferramentas

def formatar_relatorio_final(cod_consumidor: int, nome: str, status: str, detalhe: str) -> str:
    """
    Gera um relatório formatado e profissional para ser exibido ao usuário final.
    
    Args:
        cod_consumidor: ID da Unidade Consumidora
        nome: Nome do cliente
        status: Resultado da classificação (ex: NORMAL, ANOMALIA_ALTA)
        detalhe: Texto explicativo com os motivos da classificação
    """
    return f"""
    ╔══════════════════════════════════════════════════════════════════════════════╗
       RELATÓRIO TÉCNICO DE ANÁLISE DE CONSUMO
    ╚══════════════════════════════════════════════════════════════════════════════╝
    ID CONSUMIDOR: {cod_consumidor}
    CLIENTE:       {nome}
    STATUS:        {status}
    
    ANÁLISE DETALHADA:
    {detalhe}
    
    --------------------------------------------------------------------------------
    Gerado automaticamente via Agente Inteligente T2T
    """

# 2. Chamada do Agente com Múltiplas Ferramentas
try:
    print("Agente iniciado. Analisando a base completa para identificar e classificar problemas...")
    
    # O Gemini agora tem acesso a busca de dados, listagem de outliers e a nossa regra de classificação!
    response = client.models.generate_content(
        model='gemini-3.1-flash-lite-preview',
        contents="Liste os códigos dos consumidores que são outliers. Escolha um deles, busque os detalhes e use a ferramenta de classificação para me dizer se é uma anomalia real. NO FINAL, use obrigatoriamente a ferramenta formatar_relatorio_final para exibir o resultado bonitinho.",
        config=types.GenerateContentConfig(
            tools=[buscar_dados_consumidor, listar_outliers_projeto, classificar_anomalia_consumo, formatar_relatorio_final],
            temperature=0.1
        )
    )
    
    print("\n--- RELATÓRIO DO AGENTE ---")
    print(response.text)
    
except Exception as e:
    print(f"Erro na execução do Agente: {e}")

## 4.7 Bônus: Few-Shot Prompting para Geração de SQL

Muitas vezes precisamos que a IA gere queries SQL específicas para a nossa arquitetura (como o Snowflake). Para evitar que a IA 'alucine' tabelas que não existem, usamos a técnica de **Few-Shot Prompting**, onde passamos alguns exemplos de como queremos a resposta.

Neste exemplo, vamos ensinar o Gemini a traduzir linguagem natural para SQL do Snowflake focado no setor elétrico.

In [ ]:
# Usamos System Instructions para definir o papel da IA
system_instruction = """
Você é um engenheiro de dados especialista em Snowflake no setor elétrico brasileiro.
Seu papel é converter perguntas em linguagem natural para queries SQL válidas e otimizadas.
A tabela principal se chama DB_ENERGIA.PRODUCAO.TB_CONSUMIDORES.

Colunas disponíveis:
- COD_CONSUMIDOR (INT)
- NOM_CONSUMIDOR (VARCHAR)
- COD_MODALIDADE_TARIFARIA (VARCHAR) - Ex: 'B1', 'B2', 'B3'
- COD_CLASSE_CONSUMO (VARCHAR) - Ex: 'RES', 'COM', 'IND'
- VLR_CONSUMO_MEDIO_KWH (FLOAT)
- FLG_ATIVO (BOOLEAN)
"""

# Few-Shot: Passamos exemplos de pergunta -> resposta
exemplos_few_shot = """
Exemplo 1:
Pergunta: Quantos consumidores residenciais ativos nós temos?
SQL: SELECT COUNT(COD_CONSUMIDOR) FROM DB_ENERGIA.PRODUCAO.TB_CONSUMIDORES WHERE COD_CLASSE_CONSUMO = 'RES' AND FLG_ATIVO = TRUE;

Exemplo 2:
Pergunta: Qual o consumo médio por modalidade tarifária?
SQL: SELECT COD_MODALIDADE_TARIFARIA, AVG(VLR_CONSUMO_MEDIO_KWH) AS MEDIA_CONSUMO FROM DB_ENERGIA.PRODUCAO.TB_CONSUMIDORES GROUP BY COD_MODALIDADE_TARIFARIA;
"""

# A pergunta real do analista
pergunta_analista = "Liste os 10 maiores consumidores industriais inativos, mostrando o nome e o consumo médio ordenados do maior para o menor."

prompt_final = f"{exemplos_few_shot}\n\nPergunta: {pergunta_analista}\nSQL:"

try:
    print("Gerando query SQL com base no Few-Shot...")
    
    response = client.models.generate_content(
        model='gemini-2.0-flash',
        contents=prompt_final,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            temperature=0.1
        )
    )
    
    print("\n--- SQL GERADO ---")
    print(response.text)
    
except Exception as e:
    print(f"Erro: {e}")

## 5. Dicas Práticas para Analistas do Setor Elétrico

In [ ]:
dicas = [
    {
        "titulo": "Sempre inclua o domínio no prompt",
        "ruim": "Crie uma função para validar dados",
        "bom": "Crie uma função Python para validar se a modalidade tarifária (B1, B2, B3, B4, A4) é consistente com a classe de consumo (RESIDENCIAL, COMERCIAL, etc.) conforme PRODIST/ANEEL"
    },
    {
        "titulo": "Peça type hints e docstrings",
        "ruim": "Faça uma função de DEC",
        "bom": "Crie uma função calcular_dec com type hints, docstring em português descrevendo a fórmula ANEEL (DEC = Σ(Di×Ci)/Ct), e exemplos de uso no docstring"
    },
    {
        "titulo": "Especifique as colunas do DataFrame",
        "ruim": "Analise o DataFrame de consumidores",
        "bom": "Analise o DataFrame df com colunas: cod_consumidor (int), cpf_cnpj (str), cod_classe_consumo (str), vlr_consumo_medio_kwh (float). Calcule..."
    },
    {
        "titulo": "Peça tratamento de erros",
        "ruim": "Conecte ao Snowflake",
        "bom": "Crie uma classe SnowflakeConnector com context manager, tratamento de ConnectionError, e log de erros. Use variáveis de ambiente para credenciais."
    },
]

for i, dica in enumerate(dicas, 1):
    print(f"\n{'='*60}")
    print(f"Dica {i}: {dica['titulo']}")
    print(f"  RUIM: {dica['ruim']}")
    print(f"  BOM:  {dica['bom'][:100]}..." if len(dica['bom']) > 100 else f"  BOM:  {dica['bom']}")